In [17]:
from bs4 import BeautifulSoup
import requests
import urllib.parse

In [18]:
SITE_CONFIGS = {
    'vnexpress.net':{
        'title': 'h1.title-detail',
        'published-date': 'span.date',
        'location': 'p.description span.location-stamp',
        'description': 'p.description',
        'content': 'article p.Normal'
    },
    'thanhnien.vn':{
        'title': 'h1.detail-title',
        'published-date': 'div.detail-time',
        'description': 'h2.detail-sapo',
        'title-content': 'div.detail-content h2',
        'content': 'div.detail-content p'
    }
}

In [ ]:
def crawl(url: str) -> dict:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'Referer': 'https://www.google.com/'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8' 
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')
        domain = urllib.parse.urlparse(url).netloc
        if domain not in SITE_CONFIGS:
            print(f"Domain {domain} not supported.")
            return None

        config = SITE_CONFIGS[domain]

        # ========== GET TITLE ==========
        title = soup.select_one(config['title']).get_text(strip=True) if soup.select_one(config['title']) else ''

        # ========== GET PUBLISHED DATE ==========
        published_date = soup.select_one(config['published-date']).get_text(strip=True) if soup.select_one(config['published-date']) else ''

        # ========== GET DESCRIPTION ==========
        description_tag = soup.select_one(config['description'])
        description = description_tag.get_text(strip=True) if description_tag else ''

        # ========== GET LOCATION ==========
        if 'location' in config:
            location_tag = soup.select_one(config['location'])
            location = location_tag.get_text(strip=True) if location_tag else ''
            description = description.replace(location, '').strip() if location else description
        else:
            location = ''

        # ========== GET CONTENT (Đã sửa để lấy nhiều thẻ p.Normal) ==========
        content_tags = soup.select(config['content']) # Sử dụng select() thay vì select_one()
        content_text = ""
        if content_tags:
            texts = []
            for tag in content_tags:
                # Xóa các tag rác bên trong mỗi đoạn
                for trash in tag.select('table, figure, div.z-news-mini, .more-news'):
                    trash.decompose()
                
                # Giữ lại text trong thẻ <a>
                for a in tag.find_all('a'):
                    a.unwrap()
                
                texts.append(tag.get_text(strip=True))
            
            # Nối các đoạn văn lại với nhau
            content_text = "\n".join(texts)
            
        return {
            'title': title,
            'location': location,
            'published_date': published_date,
            'description': description,
            'content': content_text
        }
    except Exception as e:
        print(f"Error crawling {url}: {e}")
        return None


In [20]:
urls = [
    'https://thanhnien.vn/hinh-anh-dep-noi-san-truong-thpt-dong-ha-sau-chuong-trinh-tu-van-mua-thi-185260314110402598.htm',
    'https://vnexpress.net/merson-tottenham-tinh-sa-thai-hlv-tudor-sau-tran-gap-liverpool-5050357.html'
]
import json
for result in [crawl(url) for url in urls]:
    print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "title": "Hình ảnh đẹp nơi sân Trường THPT Đông Hà sau chương trình Tư vấn mùa thi",
  "location": "",
  "published_date": "14/03/2026 11:39 GMT+7",
  "description": "Sau phiên tư vấn buổi sáng, nhiều học sinh Trường THPT Đông Hà (phường Đông Hà, Quảng Trị) đã nán lại cùng nhau dọn dẹp vệ sinh, làm sạch khuôn viên trường, chuẩn bị đón các bạn học sinh trong phiên chiều.",
  "content": "Kết thúc phiên tư vấn buổi sáng của chương trìnhTư vấn mùa thi2026 tạiQuảng Trị, khi trời vừa ngớt mưa, một hình ảnh đẹp đã lan tỏa khắp sân Trường THPT Đông Hà (phường Đông Hà). Thay vì vội vã ra về hay tập trung tại các gian hàng, nhiều bạn trẻ đã tự giác xắn áo, chung tay nhặt rác để làm sạch không gian vừa diễn ra sự kiện.\nNhóm học sinh lớp 12 Trường THPT Đông Hà tự giác nhặt rác ngay khi phiên tư vấn buổi sáng vừa kết thúc\nẢNH: LÊ HOÀI NHÂN\nEm Nguyễn Trương Nguyên Hạo, học sinh lớp 12 Trường THPT Đông Hà, chia sẻ dù thời tiết có chút không thuận lợi khiến chỗ ngồi bị hạn chế, nhưng những thôn